# Comando usado para rodar avaliação

```
docker run --rm --gpus all \
  -v "/home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_medgemma_27b_20260318_213004/rag_prompt_config_1.5_4b_imageclef2026-rag_rag_medgemma27b_20260318_213004.csv:/app/submission.csv" \
  -v "/home/ia368/projetos/imageclef2026-rag/evaluation/caption_prediction/precomputed:/app/precomputed" \
  -v "/home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_medgemma_27b_20260318_213004:/app/output" \
  caption_prediction_evaluator valid
```

In [1]:
submission = "/home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_medgemma_27b_20260318_213004/rag_prompt_config_1.5_4b_imageclef2026-rag_rag_medgemma27b_20260318_213004.csv"

In [2]:
from pathlib import Path

# Caminho do arquivo original
input_file = Path(submission)

# (Opcional) backup
backup_file = input_file.with_suffix(input_file.suffix + ".bak")
backup_file.write_text(input_file.read_text(encoding="utf-8", errors="replace"), encoding="utf-8")

# Remove linhas vazias
lines = input_file.read_text(encoding="utf-8", errors="replace").splitlines()
clean_lines = [line for line in lines if line.strip() != ""]

# Regrava arquivo limpo
input_file.write_text("\n".join(clean_lines) + "\n", encoding="utf-8")

print(f"Arquivo limpo com sucesso!")
print(f"Linhas originais: {len(lines)}")
print(f"Linhas finais: {len(clean_lines)}")
print(f"Linhas removidas: {len(lines) - len(clean_lines)}")
print(f"Backup salvo em: {backup_file}")

Arquivo limpo com sucesso!
Linhas originais: 128
Linhas finais: 104
Linhas removidas: 24
Backup salvo em: /home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_medgemma_27b_20260318_213004/rag_prompt_config_1.5_4b_imageclef2026-rag_rag_medgemma27b_20260318_213004.csv.bak


In [3]:
from pathlib import Path

import csv

# Usa a variável `submission` da célula 1
csv_path = Path(submission)
backup_path = csv_path.with_suffix(csv_path.suffix + ".before_quote_fix.bak")

if not csv_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {csv_path}")

# 1) Lê o CSV original
with csv_path.open("r", encoding="utf-8", newline="") as f:
    rows = list(csv.reader(f))

if not rows:
    raise ValueError("CSV vazio")

# 2) Backup
backup_path.write_text(csv_path.read_text(encoding="utf-8", errors="replace"), encoding="utf-8")

# 3) Detecta cabeçalho
header = rows[0]
has_header = len(header) >= 2 and header[0].strip().lower() == "id" and header[1].strip().lower() == "caption"

data_rows = rows[1:] if has_header else rows

# 4) Reescreve no formato: ID,"Caption"
fixed_lines = []
if has_header:
    fixed_lines.append("ID,Caption")

invalid_input_rows = []

for idx, row in enumerate(data_rows, start=(2 if has_header else 1)):
    if not row or all(c.strip() == "" for c in row):
        continue

    if len(row) < 2:
        invalid_input_rows.append((idx, row))
        continue

    image_id = row[0].strip()
    caption = ",".join(row[1:]).strip()

    # Remove aspas externas se já existirem (para evitar duplicar)
    if len(caption) >= 2 and caption.startswith('"') and caption.endswith('"'):
        caption = caption[1:-1]

    # Escape de aspas internas para CSV
    caption = caption.replace('"', '""')
    fixed_lines.append(f'{image_id},"{caption}"')

# 5) Salva arquivo corrigido (sobrescreve)
csv_path.write_text("\n".join(fixed_lines) + "\n", encoding="utf-8")

# 6) Validação final do formato da coluna Caption
with csv_path.open("r", encoding="utf-8", newline="") as f:
    raw_lines = f.read().splitlines()

start_idx = 1 if has_header else 0
invalid_after_fix = []

for line_number, line in enumerate(raw_lines[start_idx:], start=start_idx + 1):
    if not line.strip():
        continue
    if "," not in line:
        invalid_after_fix.append((line_number, line, "sem separador ','"))
        continue

    _, raw_caption = line.split(",", 1)

    raw_caption = raw_caption.strip()

    if not (len(raw_caption) >= 2 and raw_caption.startswith('"') and raw_caption.endswith('"')):
        invalid_after_fix.append((line_number, line, "Caption não está entre aspas duplas"))


print(f"Arquivo corrigido: {csv_path}")
print(f"Backup: {backup_path}")
print(f"Linhas de entrada inválidas (ignoradas): {len(invalid_input_rows)}")
print(f"Linhas inválidas após correção: {len(invalid_after_fix)}")

if invalid_input_rows:
    print("\nExemplos de linhas de entrada inválidas (até 5):")
    for ln, row in invalid_input_rows[:5]:
        print(f"- linha {ln}: {row}")

if invalid_after_fix:
    print("\nExemplos de linhas inválidas após correção (até 5):")
    for ln, content, reason in invalid_after_fix[:5]:
        print(f"- linha {ln}: {reason}")
        print(f"  {content[:200]}{'...' if len(content) > 200 else ''}")
else:
    print("\n✅ Todas as captions ficaram no formato \"caption text here\".")


Arquivo corrigido: /home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_medgemma_27b_20260318_213004/rag_prompt_config_1.5_4b_imageclef2026-rag_rag_medgemma27b_20260318_213004.csv
Backup: /home/ia368/projetos/imageclef2026-rag/artifacts/results/rag_medgemma_27b_20260318_213004/rag_prompt_config_1.5_4b_imageclef2026-rag_rag_medgemma27b_20260318_213004.csv.before_quote_fix.bak
Linhas de entrada inválidas (ignoradas): 0
Linhas inválidas após correção: 90

Exemplos de linhas inválidas após correção (até 5):
- linha 2: Caption não está entre aspas duplas
  ImageCLEFmedical_Caption_2026_valid_0,"MRI of the brain showing the pituitary gland and surrounding structures. The corpus callosum appears slightly compressed. There is mild hypointensity in the T1-w...
- linha 3: sem separador ','
  (Note: The provided captions are for reference only and do not reflect the actual content of the image.)"
- linha 4: Caption não está entre aspas duplas
  ImageCLEFmedical_Caption_2026_valid_1,"thoug

In [ ]:
# outro ponto que deu problema: contem , no valor da caption